# Domain 4: Prompt Engineering & Structured Output
## Task Statement 4.2: Apply Few-Shot Prompting and Multi-Instance Review Architectures
**Certification Weight**: 18% Exam Weight | **Standard**: Claude 4.5 & Extended Thinking

## 🎯 Exam Objective
Understand how to craft targeted **few-shot examples with reasoning** to resolve edge cases, and design **multi-instance review architectures** that eliminate author confirmation bias.

---

## 🏗️ Architecture: Multi-Instance Independent Review Pipeline

```mermaid
flowchart TD
    TaskInput["Raw Task Input / Invoices"] --> GeneratorInstance["Instance 1: Generator<br>(claude-sonnet-4-5)"]
    
    GeneratorInstance --> FewShotPrompt["Prompt with 2-4 Targeted Few-Shot Examples<br>(demonstrating discrepancy handling)"]
    FewShotPrompt --> CandidateArtifact["Initial Generated Candidate Artifact<br>(e.g. Extracted JSON / Draft PR)"]
    
    CandidateArtifact --> ReviewerInstance["Instance 2: Independent Reviewer<br>(Fresh Context, Zero Generator History)"]
    
    ReviewerInstance --> BlindEvaluation{"Independent Evaluation<br>against Strict Rubric"}
    
    BlindEvaluation -->|"Discrepancy / Bug Found"| FeedbackLoop["Provide Structured Bug Report<br>Back to Generator"]
    FeedbackLoop --> GeneratorInstance
    
    BlindEvaluation -->|"Clean / Verified"| ApprovedOutput["Final Verified Output Payload"]
```

---

## ⚠️ Critical Exam Distinctions & Anti-Patterns

| Prompting / Review Architecture | Recommended Implementation | Exam Trap to Reject |
|---|---|---|
| **Few-Shot Example Selection** | Curate **2 to 4 high-yield examples** specifically targeting borderline and ambiguous edge cases with reasoning. | **TRAP**: Bloating the system prompt with 30+ trivial happy-path examples, consuming unnecessary context window tokens. |
| **Review Isolation** | Use an **independent Claude instance** with zero access to the generator's internal thinking blocks or rationale. | **TRAP**: Asking the same generator agent to "self-review" its own output in the same conversation thread (prone to confirmation bias). |
| **Few-Shot Example Structure** | Include input, step-by-step reasoning justification, and final structured output. | **TRAP**: Providing only raw inputs and outputs without explaining the decision rule for edge cases. |

---

## 💡 Architectural Decision Matrix

| Review Pattern | Independence Level | Confirmation Bias | Ideal Use Case |
|---|---|---|---|
| **Single-Agent Self-Review** | Zero (same context thread) | High | Trivial typo and grammar checks. |
| **Multi-Turn Critic** | Partial (same session, subsequent turn) | Moderate | Code formatting and style alignment. |
| **Multi-Instance Independent Review** | Complete (isolated fresh session) | Low / None | Security auditing, legal extraction, compliance verification. |

---

---

### 🧠 Claude 4.5 Extended Thinking & Parameter Standards (October 2026)
- **Model Standard**: Use `claude-sonnet-4-5` or `claude-haiku-4-5`.
- **Temperature Omission**: When `thinking` is enabled (`thinking={"type": "enabled", "budget_tokens": ...}`), **do not provide `temperature`**, `top_p`, or `top_k`. The Anthropic API rejects custom sampling values with an HTTP `400 Bad Request` error.
- **Thinking Continuity**: In multi-turn loops, assistant reasoning blocks (`type: "thinking"`) must be returned **unmodified with their cryptographic signature** in subsequent turns. Stripping thinking blocks causes an API `400 Bad Request` error.
- **Case Facts Preservation**: Immutable transactional data (Order IDs, customer accounts, amounts) must be maintained in a dedicated Case Facts block outside conversation summaries.

In [ ]:
# Step 1: Environment & Setup
import json
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(" Environment initialized for TS 4.2 Few-Shot & Multi-Instance.")

### 📚 Step 2: Few-Shot Prompt Block & Multi-Instance Pipeline
We implement the few-shot template with thinking blocks and the multi-instance aggregator.

In [ ]:
# Step 2: Few-Shot & Multi-Instance Logic
@dataclass
class FewShotExample:
    input_text: str
    thinking: str
    output_json: Dict[str, Any]

FEW_SHOT_EXTRACTION_EXAMPLES = [
    FewShotExample(
        input_text="Invoice from CloudHost Inc: 1x Server $100, 2x IP Addr $10, Stated Total $120.",
        thinking=(
            "1. Server: 1 * 100 = 100. "
            "2. IP: 2 * 10 = 20. "
            "3. Calculated sum = 120. Stated total = 120. Arithmetic matches perfectly."
        ),
        output_json={"vendor": "CloudHost Inc", "calculated_total": 120.0, "status": "VALID"}
    ),
    FewShotExample(
        input_text="Receipt: Lunch $45, Tip $10, Total stated $60.",
        thinking=(
            "1. Lunch = 45, Tip = 10. "
            "2. Expected total = 55. Stated total = 60. Math mismatch of $5. Flag arithmetic anomaly."
        ),
        output_json={"vendor": "Unknown", "calculated_total": 55.0, "status": "ARITHMETIC_MISMATCH"}
    ),
]

def render_few_shot_prompt_block(examples: List[FewShotExample]) -> str:
    """Renders input, thinking, and output blocks for few-shot guidance."""
    parts = ["### FEW-SHOT REASONING EXAMPLES"]
    for i, ex in enumerate(examples, start=1):
        parts.append(
            f"--- EXAMPLE {i} ---\n"
            f"<input>\n{ex.input_text}\n</input>\n"
            f"<thinking>\nArchitectural Reasoning: {ex.thinking}\n</thinking>\n"
            f"<output>\n{json.dumps(ex.output_json, indent=2)}\n</output>"
        )
    return "\n\n".join(parts)


class MultiInstanceReviewPipeline:
    @staticmethod
    def prepare_reviewer_payload(code_snippet: str, review_checklist: List[str]) -> Dict[str, Any]:
        """Creates an isolated reviewer prompt ensuring no shared conversational state."""
        checklist_str = "\n".join(f"- {item}" for item in review_checklist)
        return {
            "role": "user",
            "content": (
                "You are an independent, isolated code auditor. "
                "Evaluate this code strictly against the checklist without assuming prior context:\n"
                f"CHECKLIST:\n{checklist_str}\n\n"
                f"CODE:\n{code_snippet}"
            )
        }

    @staticmethod
    def aggregate_independent_reviews(verdicts: List[Dict[str, Any]]) -> Dict[str, Any]:
        """Aggregates verdicts across independent instances via majority consensus."""
        approvals = sum(1 for v in verdicts if v.get("approved", False))
        consensus_approved = (approvals >= len(verdicts) / 2)
        return {
            "total_instances": len(verdicts),
            "approvals": approvals,
            "consensus_approved": consensus_approved,
        }

print(" Few-shot and Multi-instance engines ready.")

### 🧪 Step 3: Interactive Simulation
1. Render few-shot prompt block with `<thinking>` tags.
2. Simulate 3 independent reviewer instances and aggregate consensus.

In [ ]:
# Step 3: Simulation
# 1. Render few-shot prompt block
prompt_block = render_few_shot_prompt_block(FEW_SHOT_EXTRACTION_EXAMPLES)
print(prompt_block[:350] + "\n... [truncated]")

# 2. Multi-instance independent reviews
reviewer_verdicts = [
    {"instance_id": "inst_1", "approved": True, "findings": []},
    {"instance_id": "inst_2", "approved": True, "findings": []},
    {"instance_id": "inst_3", "approved": False, "findings": ["Minor typing issue"]},
]
consensus = MultiInstanceReviewPipeline.aggregate_independent_reviews(reviewer_verdicts)
print("\nMulti-Instance Consensus Result:", consensus)

### 🔬 Step 4: Verification Assertions
We assert that:
1. Few-shot prompt includes explicit `<thinking>` blocks.
2. Independent instance payloads do not share history.
3. Majority consensus correctly resolves 2 approvals out of 3 as approved.

In [ ]:
# Step 4: Verification Checks
assert "<thinking>" in prompt_block
assert "Architectural Reasoning:" in prompt_block
assert "<output>" in prompt_block

payload = MultiInstanceReviewPipeline.prepare_reviewer_payload("x = 1", ["Check style"])
assert "independent, isolated code auditor" in payload["content"]

assert consensus["consensus_approved"] is True
assert consensus["approvals"] == 2
assert consensus["total_instances"] == 3

print(" All verification assertions passed! Few-shot & multi-instance verified.")

### 💡 CCAR-F Exam Golden Rules
- **Rule 1**: Include explicit `<thinking>` blocks in few-shot demonstrations to anchor Claude's reasoning chain.
- **Rule 2**: Use independent instances without shared memory to eliminate confirmation and sequential review bias.
- **Rule 3**: Combine independent instance outputs using majority-voting consensus.